# 03 — Pair Feature Engineering (Phase 5)

Builds the full feature matrix (`src/features.py`) for the candidate pairs
from notebook 02 and inspects feature distributions / class separation.
Also builds the training-data-only IDF (rarity) tables. Run on AWS after
notebook 02 (same sampled Source-1 ids, so results compose).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src import config, features, retrieval, train as train_module
from src.data_loader import load_normalized_source, load_ground_truth_exploded
from src.inference import VectorizedOtherSide, candidates_for_chunk_and_source
from src.labeling import label_candidates_fast
from src.utils import timer


In [ ]:
s1_norm = load_normalized_source("train", "source1")
s2_norm = load_normalized_source("train", "source2")
s3_norm = load_normalized_source("train", "source3")

sample_ids = train_module.sample_source1_ids(s1_norm["entity_id"].tolist(), config.EXPERIMENT_SAMPLE_SIZE, config.RANDOM_SEED)
s1_sample = s1_norm[s1_norm["entity_id"].isin(set(sample_ids))].reset_index(drop=True)

name_idf = features.build_idf_table(s1_norm["name_tokens"].tolist() + s2_norm["name_tokens"].tolist() + s3_norm["name_tokens"].tolist())
addr_idf = features.build_idf_table(s1_norm["address_tokens"].tolist() + s2_norm["address_tokens"].tolist() + s3_norm["address_tokens"].tolist())
len(name_idf), len(addr_idf)


In [ ]:
name_vec = retrieval.fit_field_vectorizer([s1_norm["name_alnum"], s2_norm["name_alnum"], s3_norm["name_alnum"]])
addr_vec = retrieval.fit_field_vectorizer([s1_norm["address_alnum"], s2_norm["address_alnum"], s3_norm["address_alnum"]])
s2_side = VectorizedOtherSide(s2_norm, name_vec, addr_vec)
s3_side = VectorizedOtherSide(s3_norm, name_vec, addr_vec)

with timer("candidates"):
    cand_s2 = candidates_for_chunk_and_source(s1_sample, s2_side, name_vec, addr_vec, config.MAX_BLOCK_SIZE, config.TFIDF_TOP_K)
    cand_s3 = candidates_for_chunk_and_source(s1_sample, s3_side, name_vec, addr_vec, config.MAX_BLOCK_SIZE, config.TFIDF_TOP_K)


In [ ]:
rule_cols = [c for c in cand_s2.columns if c.startswith("found_by_")] + ["n_blocking_rules"]

merged_s2 = features.merge_pair_fields(cand_s2[["entity_id_s1", "entity_id_other"]], s1_sample, s2_norm)
merged_s2 = merged_s2.assign(tfidf_score_name=cand_s2["tfidf_score_name"].to_numpy(), tfidf_score_address=cand_s2["tfidf_score_address"].to_numpy())
with timer("compute features (Source-2 side)"):
    feat_s2 = features.compute_pair_features(merged_s2, name_idf, addr_idf, rule_flags=cand_s2[rule_cols])

merged_s3 = features.merge_pair_fields(cand_s3[["entity_id_s1", "entity_id_other"]], s1_sample, s3_norm)
merged_s3 = merged_s3.assign(tfidf_score_name=cand_s3["tfidf_score_name"].to_numpy(), tfidf_score_address=cand_s3["tfidf_score_address"].to_numpy())
with timer("compute features (Source-3 side)"):
    feat_s3 = features.compute_pair_features(merged_s3, name_idf, addr_idf, rule_flags=cand_s3[rule_cols])

feat_all = pd.concat([feat_s2, feat_s3], ignore_index=True)
feat_all.shape


In [ ]:
truth_exploded = load_ground_truth_exploded()
labels = label_candidates_fast(feat_all, truth_exploded)
labels.value_counts()


## Feature separation: mean value by label (sanity check that features carry signal)

In [ ]:
feat_cols = features.feature_columns(feat_all)
comparison = feat_all[feat_cols].assign(label=labels.values).groupby("label").mean().T
comparison["gap"] = comparison[1] - comparison[0]
comparison.sort_values("gap", ascending=False).head(20)


Features with a large positive gap (higher for true matches than for hard
negatives) are the ones actually carrying discriminative signal for this
dataset -- expect the exact-match / high fuzzy-ratio / rare-token-overlap
features near the top. Save `feat_all`/`labels` (e.g. to parquet) here if
you want to hand them to notebook 04 without recomputing.